# 01 Explore raw FPL data

Goal: answer the open questions for `DATA.md` (see CLAUDE.md).

In [1]:
import gzip, json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"

def load(path):
    with gzip.open(path, "rt") as f:
        return json.load(f)

sorted(p.name for p in RAW.iterdir())

['daily', 'element_summary', 'gameweeks']

In [2]:
# Load raw JSON into DataFrames (open them from the Variables panel in Data Wrangler)

# Latest daily snapshot: current state of players, teams, gameweeks, fixtures
latest = sorted((RAW / "daily").iterdir())[-1]
boot = load(latest / "bootstrap-static.json.gz")

players = pd.DataFrame(boot["elements"])        # one row per player, CURRENT state (leaks if used for past GWs)
teams = pd.DataFrame(boot["teams"])
positions = pd.DataFrame(boot["element_types"])
gameweeks = pd.DataFrame(boot["events"])
fixtures = pd.DataFrame(load(latest / "fixtures.json.gz"))

# Per-gameweek live stats: one row per player per GW (double GWs are summed)
rows = []
for p in sorted((RAW / "gameweeks").glob("gw*_live.json.gz")):
    gw = int(p.name[2:4])
    for e in load(p)["elements"]:
        rows.append({"gw": gw, "id": e["id"], **e["stats"]})
live = pd.DataFrame(rows)

# Per-player match history from the latest element_summary pull: one row per match
summary_dir = sorted((RAW / "element_summary").iterdir())[-1]
history = pd.DataFrame(
    [match for p in summary_dir.glob("*.json.gz") for match in load(p)["history"]]
)

print(f"snapshot: {latest.name}, element_summary: {summary_dir.name}")
for name in ["players", "teams", "positions", "gameweeks", "fixtures", "live", "history"]:
    print(f"{name:10} {globals()[name].shape}")

snapshot: 2026-10-07T2216Z, element_summary: 2026-10-07
players    (667, 109)
teams      (20, 22)
positions  (4, 13)
gameweeks  (38, 29)
fixtures   (380, 17)
live       (3216, 31)
history    (3216, 41)
